# 03 - Analisis territorial

Exploracion de FIRMS, ZARI y combustible para la camara simulada de Tejeda. Este notebook usa las funciones de `geo_pipeline/` y exporta las capas GeoJSON del dashboard.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from geo_pipeline import CAMERA, GC_BBOX, RADIO_FIRMS_KM
from geo_pipeline.firms_loader import cargar_firms, densidad_hotspots
from geo_pipeline.zari_loader import cargar_zari, punto_en_zari
from geo_pipeline.combustible_loader import cargar_combustible_gc, combustible_en_punto
from geo_pipeline.risk_index import calcular_indice_riesgo
from geo_pipeline.geojson_export import exportar_geojsons


## 1. FIRMS

Carga de puntos historicos filtrados para Gran Canaria y resumen temporal.


In [ ]:
firms = cargar_firms(ROOT / "Datasets Territoriales" / "NASA Firms", GC_BBOX)
firms.head()


In [ ]:
densidad = densidad_hotspots(firms, CAMERA["lat"], CAMERA["lon"], RADIO_FIRMS_KM)
densidad


In [ ]:
ax = firms.plot(figsize=(7, 7), markersize=4, alpha=0.45)
ax.set_title("Hotspots historicos FIRMS - Gran Canaria")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")


## 2. ZARI

Inspeccion de zonas de alto riesgo y comprobacion del punto de camara.


In [ ]:
zari = cargar_zari(ROOT / "Datasets Territoriales" / "ZARI")
zari_gc = zari[zari["ISLA"].astype(str).str.upper().str.contains("GRAN CANARIA", na=False)]
zari_gc[["ZONA", "HECTARES", "ISLA", "B__O__C_"]].head(10)


In [ ]:
punto_zari = punto_en_zari(zari, CAMERA["lat"], CAMERA["lon"])
punto_zari


In [ ]:
ax = zari_gc.plot(figsize=(7, 7), alpha=0.25, edgecolor="red")
ax.scatter([CAMERA["lon"]], [CAMERA["lat"]], c="black", s=40)
ax.set_title("ZARI Gran Canaria y camara simulada")


## 3. Combustible

Carga del modelo de combustible canario y analisis del radio de 5 km.


In [ ]:
comb = cargar_combustible_gc(ROOT / "Datasets Territoriales" / "Modelos de Combustible por Isla" / "GRAN CANARIA")
comb[["mc", "codigo", "peso", "descripcion"]].head()


In [ ]:
combustible_radio = combustible_en_punto(comb, CAMERA["lat"], CAMERA["lon"], 5.0)
combustible_radio


In [ ]:
comb.groupby("codigo").size().sort_values(ascending=False).head(15).plot(kind="bar", figsize=(9, 4), title="Poligonos por modelo de combustible")


## 4. Indice de riesgo

Calculo del indice compuesto sin AEMET en este notebook base; el dashboard incorpora meteo real si hay API key.


In [ ]:
meteo_simulada = {"viento_vel": 0, "error": "sin AEMET en notebook"}
indice = calcular_indice_riesgo(punto_zari["dentro"], combustible_radio["peso"], densidad, meteo_simulada)
indice


In [ ]:
import pandas as pd
pd.Series(indice["componentes"]).plot(kind="bar", ylim=(0, 1), title="Componentes normalizados del riesgo")


## 5. Exportacion GeoJSON

Genera las capas estaticas utilizadas por `dashboard/map_builder.py`.


In [ ]:
exportar_geojsons(ROOT)
